# CellLineSelector — Exploratory Data Analysis

**Goal of this notebook:** give every teammate — whether you come from biology, data science, or software engineering — a complete, walk-through tour of the raw data we have to work with, and *why* each dataset matters for picking the right cancer cell line for a drug study.

## The big picture: why cell lines at all?

Pharmaceutical research (here, for AstraZeneca) can't test a new drug on humans first. Instead, scientists grow **cell lines** — populations of cells, usually derived from a patient's tumour, that can be grown indefinitely in a lab dish. Each cell line is like a little "avatar" of a real tumour: it carries a snapshot of that tumour's genetics, gene activity, and metabolism.

There are **thousands** of cancer cell lines available (e.g. from the Cancer Cell Line Encyclopedia, CCLE, and the Broad Institute's DepMap project). The challenge: **which cell line(s) should a researcher pick** to test a drug aimed at, say, a specific mutated gene or pathway?

Our job is to build a `CellLineSelector` pipeline that, given a research question (e.g. "a drug targeting gene X in lung cancer"), recommends a **top-10 shortlist of cell lines** with a clear justification — "why this cell line and not another".

## The four layers of biological data

To justify a recommendation we need to look at a cell line from multiple biological "layers", each captured by a different file:

| Layer | What it tells us | Folder |
|---|---|---|
| **Gene expression (transcriptomics)** | Which genes are "switched on" and how strongly, in this cell line | `gene_expression/` |
| **Gene properties (genomics)** | Structural/genetic abnormalities — mutations, gene fusions | `gene_properties/` |
| **Nomenclature / metadata** | Who is this cell line? What cancer type, tissue, patient details | `nomenclature/` |
| **Non-gene expression (other omics)** | Metabolite levels, microRNAs, and chromosomal instability scores | `non_gene_expression/` |

Think of it like a medical chart for each cell line: the **nomenclature** files are the patient ID card, **gene expression** is like a blood test showing which systems are active, **gene properties** is the genetic test result, and **non-gene expression** is everything else (metabolic panel, hormone levels, etc.).

Below, we open every one of the 14 raw files, look at their shape and structure, and connect them via the identifiers that link a row back to a specific cell line.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

RAW = '../../data/raw'

GENE_EXPR = f'{RAW}/gene_expression'
GENE_PROP = f'{RAW}/gene_properties'
NOMEN = f'{RAW}/nomenclature'
NON_GENE_EXPR = f'{RAW}/non_gene_expression'

---
# 1. Nomenclature — "Who is this cell line?"

We start here, *out of file-number order*, because these tables are the **Rosetta Stone** of the whole project. Every other dataset (gene expression, mutations, metabolomics...) identifies cell lines using one of several different ID systems — `ACH-xxxxxx` (DepMap), `CVCL_xxxx` (Cellosaurus), plain names like `MCF7`, or GEO accession codes (`GSMxxxxx`). The nomenclature tables tell us how these IDs map to each other, and give us human-readable context: what cancer, what tissue, what patient.

Without this layer, the rest of the data is just numbers attached to cryptic codes — this is what turns it into biology.

## 1.1 — File 9: DepMap Sample Info

**What it is:** The master catalogue of cell lines used by DepMap (Broad Institute's Cancer Dependency Map project). Each row is one cell line.

**Why it matters:** This is our primary "directory" — it gives us the cancer type (`primary_disease`), tissue of origin (`lineage`), sex, age, and whether the sample was a primary tumour or a metastasis. When we eventually justify a recommendation ("why this cell line?"), this table supplies the clinical/biological story.

In [2]:
sample_info = pd.read_csv(f'{NOMEN}/9_DepMap_sample_info.csv')
sample_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 1840 entries, 0 to 1839
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   DepMap_ID                   1840 non-null   str    
 1   cell_line_name              1748 non-null   str    
 2   stripped_cell_line_name     1839 non-null   str    
 3   CCLE_Name                   1836 non-null   str    
 4   alias                       113 non-null    str    
 5   COSMICID                    981 non-null    float64
 6   sex                         1738 non-null   str    
 7   source                      1793 non-null   str    
 8   RRID                        1818 non-null   str    
 9   WTSI_Master_Cell_ID         980 non-null    float64
 10  sample_collection_site      1833 non-null   str    
 11  primary_or_metastasis       1198 non-null   str    
 12  primary_disease             1840 non-null   str    
 13  Subtype                     1696 non-null   

In [3]:
sample_info.head()

,DepMap_ID,cell_line_name,stripped_cell_line_name,CCLE_Name,alias,COSMICID,sex,source,RRID,WTSI_Master_Cell_ID,...,lineage_sub_subtype,lineage_molecular_subtype,default_growth_pattern,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,Cellosaurus_NCIt_disease,Cellosaurus_NCIt_id,Cellosaurus_issues
0,ACH-000016,SLR 21,SLR21,SLR21_KIDNEY,NaN,NaN,NaN,Academic lab,CVCL_V607,NaN,...,NaN,NaN,NaN,NaN,NaN,PT-JnARLB,NaN,Clear cell renal cell carcinoma,C4033,NaN
1,ACH-000032,MHH-CALL-3,MHHCALL3,MHHCALL3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,NaN,Female,DSMZ,CVCL_0089,NaN,...,b_cell,NaN,NaN,NaN,NaN,PT-p2KOyI,NaN,Childhood B acute lymphoblastic leukemia,C9140,NaN
2,ACH-000033,NCI-H1819,NCIH1819,NCIH1819_LUNG,NaN,NaN,Female,Academic lab,CVCL_1497,NaN,...,NSCLC_adenocarcinoma,NaN,NaN,NaN,NaN,PT-9p1WQv,NaN,Lung adenocarcinoma,C3512,NaN
3,ACH-000043,Hs 895.T,HS895T,HS895T_FIBROBLAST,NaN,NaN,Female,ATCC,CVCL_0993,NaN,...,NaN,NaN,2D: adherent,NaN,NaN,PT-rTUVZQ,NaN,Melanoma,C3224,NaN
4,ACH-000049,HEK TE,HEKTE,HEKTE_KIDNEY,NaN,NaN,NaN,Academic lab,CVCL_WS59,NaN,...,NaN,NaN,NaN,immortalized,NaN,PT-qWYYgr,NaN,NaN,NaN,No information is available about this cell li...


In [4]:
# How many distinct cell lines do we have, and how are they distributed across tissues ("lineages")?
print(f"Unique cell lines: {sample_info['DepMap_ID'].nunique()}")
print(f"Unique tissue lineages: {sample_info['lineage'].nunique()}")
sample_info['lineage'].value_counts().head(10)

Unique cell lines: 1840
Unique tissue lineages: 30


lineage
lung                      274
blood                     141
skin                      120
lymphocyte                110
central_nervous_system    109
breast                     86
colorectal                 85
bone                       79
upper_aerodigestive        79
ovary                      74
Name: count, dtype: int64

**Reading the output:** ~1,840 cell lines spanning 30 tissue types ("lineages"). Lung, blood, and skin cancers are the most heavily represented — this reflects how common these cancers are in research collections, not necessarily in the population. Knowing this distribution matters: if a researcher asks for a rare cancer type, we may have very few candidate cell lines to choose from.

## 1.2 — File 8: DepMap Omics Profiles

**What it is:** A lookup table that links a `ModelID` (= the `DepMap_ID` / `ACH-xxxxxx` from file 9) to one or more **sequencing profile IDs** (`PR-xxxxxx`), and tells us what *type* of experiment that profile is (`wes` = whole exome sequencing, `wgs` = whole genome sequencing, `rna` = RNA sequencing).

**Why it matters:** Several of our gene-level files (mutations, fusions, global signatures) are indexed by `ProfileID` (`PR-xxxxxx`), not by the cell line ID directly. This table is the *bridge* that lets us join those files back to a specific cell line.

In [5]:
omics_profiles = pd.read_csv(f'{NOMEN}/8_DepMap_OmicsProfiles.csv')
omics_profiles.info()

<class 'pandas.DataFrame'>
RangeIndex: 3830 entries, 0 to 3829
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   ProfileID       3830 non-null   str  
 1   ModelCondition  3830 non-null   str  
 2   ModelID         3830 non-null   str  
 3   Datatype        3830 non-null   str  
 4   WESKit          1860 non-null   str  
dtypes: str(5)
memory usage: 149.7 KB


In [6]:
omics_profiles.head()

,ProfileID,ModelCondition,ModelID,Datatype,WESKit
0,PR-00UtU3,MC-001131-kkJv,ACH-001131,wgs,NaN
1,PR-01r7OM,MC-000957-Yckn,ACH-000957,rna,NaN
2,PR-02XmLG,MC-002785-qo9e,ACH-002785,rna,NaN
3,PR-04VvBz,MC-001289-BpdI,ACH-001289,wes,ICE
4,PR-09gmEI,MC-000520-YIm7,ACH-000520,rna,NaN


In [7]:
# A cell line can have multiple profiles (one for WES, one for RNA, etc.)
print(f"Unique ModelIDs: {omics_profiles['ModelID'].nunique()}")
print(f"Unique ProfileIDs: {omics_profiles['ProfileID'].nunique()}")
omics_profiles['Datatype'].value_counts()

Unique ModelIDs: 1822
Unique ProfileIDs: 3830


Datatype
wes    1860
rna    1495
wgs     475
Name: count, dtype: int64

**Reading the output:** Most cell lines have a `wes` (exome) and `rna` profile, and a smaller subset also has `wgs` (whole genome). This is important to know: not every cell line has every type of data available, so our pipeline needs to handle missing layers gracefully.

## 1.3 — File 7: Cellosaurus

**What it is:** [Cellosaurus](https://www.cellosaurus.org/) is a huge, community-curated *encyclopedia* of cell lines from all of biomedical research (not just cancer, and not just DepMap). Each entry has a unique `CVCL_xxxx` accession, plus rich free-text metadata: synonyms, the disease it models, species, sex/age of the donor, and STR (DNA fingerprint) profiles used to verify a cell line's identity.

**Why it matters:** This is the largest and most authoritative naming registry. It helps us resolve cell line *synonyms* (the same cell line is often called different things in different datasets) and gives extra QC information (e.g. STR profiling flags contaminated/misidentified cell lines — a real problem in cancer research).

In [8]:
cellosaurus = pd.read_csv(f'{NOMEN}/7_cellosaurus.csv')
cellosaurus.info()

<class 'pandas.DataFrame'>
RangeIndex: 152231 entries, 0 to 152230
Data columns (total 17 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   Identifier (cell line name)     152230 non-null  str  
 1   Accession (CVCL_xxxx)           152231 non-null  str  
 2   Secondary accession number(s)   542 non-null     str  
 3   Synonyms                        72386 non-null   str  
 4   Cross-references                150430 non-null  str  
 5   References identifiers          83089 non-null   str  
 6   Web pages                       11630 non-null   str  
 7   Comments                        150484 non-null  str  
 8   STR profile data                8734 non-null    str  
 9   Diseases                        70990 non-null   str  
 10  Species of origin               152231 non-null  str  
 11  Hierarchy                       55937 non-null   str  
 12  Originate from same individual  15554 non-null   str  


In [9]:
cellosaurus.head(3)

,Identifier (cell line name),Accession (CVCL_xxxx),Secondary accession number(s),Synonyms,Cross-references,References identifiers,Web pages,Comments,STR profile data,Diseases,Species of origin,Hierarchy,Originate from same individual,Sex of cell,Age of donor at sampling,Category,Date (entry history)
0,#132 PC3-1-SC-E8,CVCL_B0T9,NaN,Z48-5MG-70,Wikidata; Q108819335,Patent=EP0501779A1;,NaN,Group: Patented cell line. || Registration: In...,NaN,NaN,NCBI_TaxID=10090; ! Mus musculus (Mouse),CVCL_D145 ! HL-1 Friendly Myeloma-653,NaN,NaN,NaN,Hybridoma,Created: 23-09-21; Last updated: 30-01-24; Ver...
1,#132 PL12 SC-D1,CVCL_B0T8,NaN,Z48-5MG-63,Wikidata; Q108819336,Patent=EP0501779A1;,NaN,Group: Patented cell line. || Registration: In...,NaN,NaN,NCBI_TaxID=10090; ! Mus musculus (Mouse),CVCL_D145 ! HL-1 Friendly Myeloma-653,NaN,NaN,NaN,Hybridoma,Created: 23-09-21; Last updated: 30-01-24; Ver...
2,#15310-LN,CVCL_E548,NaN,15310-LN; TER461; TER-461; Ter 461; TER479; TE...,dbMHC; 48439 || ECACC; 94050311 || IHW; IHW093...,NaN,http://pathology.ucla.edu/workfiles/360cx.pdf ...,Part of: 12th International Histocompatibility...,NaN,NaN,NCBI_TaxID=9606; ! Homo sapiens (Human),NaN,NaN,Female,Age unspecified,Transformed cell line,Created: 22-10-12; Last updated: 30-01-24; Ver...


In [10]:
print(f"Total Cellosaurus entries: {len(cellosaurus):,}")
print(f"Unique species: {cellosaurus['Species of origin'].nunique()}")
cellosaurus['Category'].value_counts()

Total Cellosaurus entries: 152,231
Unique species: 835


Category
Transformed cell line                   50625
Cancer cell line                        36263
Embryonic stem cell                     16478
Induced pluripotent stem cell           16381
Finite cell line                        12263
Hybridoma                               11013
Spontaneously immortalized cell line     6449
Telomerase immortalized cell line         833
Hybrid cell line                          747
Conditionally immortalized cell line      375
Factor-dependent cell line                276
Undefined cell line type                  233
Somatic stem cell                         229
Stromal cell line                          66
Name: count, dtype: int64

**Reading the output:** Cellosaurus covers ~hundreds of thousands of entries across many species and categories (cancer cell lines, hybridomas, stem cells, etc.). For our project we only care about a small slice — the human cancer cell lines that overlap with DepMap/CCLE/HPA/GEO. We'll use this table mainly to **cross-walk names**, not as a primary data source.

## 1.4 — File 11: HPA Cell Line Description

**What it is:** A small reference table from the [Human Protein Atlas](https://www.proteinatlas.org/) (HPA) describing each cell line they profiled: the disease, disease subtype, matching Cellosaurus ID, and whether the sample is primary or metastatic tissue.

**Why it matters:** This is the metadata companion to File 1 (HPA RNA expression). It tells us what disease each HPA cell line represents, so we can connect HPA's expression measurements to a cancer type.

In [11]:
hpa_desc = pd.read_csv(f'{NOMEN}/11_hpa_rna_celline_description.tsv', sep='\t')
hpa_desc.info()

<class 'pandas.DataFrame'>
RangeIndex: 1206 entries, 0 to 1205
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Cell line               1206 non-null   str  
 1   Disease                 1206 non-null   str  
 2   Disease subtype         1047 non-null   str  
 3   Cellosaurus ID          1198 non-null   str  
 4   Patient                 1101 non-null   str  
 5   Primary/Metastasis      867 non-null    str  
 6   Sample collection site  1138 non-null   str  
dtypes: str(7)
memory usage: 66.1 KB


In [12]:
hpa_desc.head()

,Cell line,Disease,Disease subtype,Cellosaurus ID,Patient,Primary/Metastasis,Sample collection site
0,143B,Bone cancer,Osteosarcoma,CVCL_2270,13,Primary,bone
1,22Rv1,Prostate cancer,Adenocarcinoma,CVCL_1045,Male,primary,prostate
2,23132/87,Gastric cancer,Adenocarcinoma,CVCL_1046,"Male, 72",primary,stomach
3,253J,Bladder cancer,Carcinoma,CVCL_7935,"Male, 53",metastasis,lymph node
4,253J-BV,Bladder cancer,Carcinoma,CVCL_7937,"Male, 53",metastasis,lymph node


In [13]:
print(f"Cell lines described: {hpa_desc['Cell line'].nunique()}")
print(f"Distinct diseases: {hpa_desc['Disease'].nunique()}")
hpa_desc['Disease'].value_counts().head(10)

Cell lines described: 1206
Distinct diseases: 30


Disease
Lung cancer          232
Leukemia              93
Brain cancer          80
Lymphoma              76
Non-cancerous         63
Colorectal cancer     63
Skin cancer           62
Breast cancer         62
Ovarian cancer        59
Pancreatic cancer     46
Name: count, dtype: int64

## 1.5 — File 10: GEO Info

**What it is:** Metadata for samples from the [Gene Expression Omnibus](https://www.ncbi.nlm.nih.gov/geo/) (GEO) — a public repository where labs around the world deposit gene expression experiments. Each row is one sample (`GSM` accession) with its experimental context (cell line, disease, originating lab, platform used).

**Why it matters:** GEO data (File 3) gives us *more* expression measurements, often for cell lines or conditions not covered by DepMap/HPA. The catch is that GEO is crowd-sourced from thousands of independent experiments, so cell line names are messy/inconsistent — this file includes columns like `Matching_Type` and `cell_line_Trimmed` that show attempts to clean and match these names back to standard identifiers (e.g. Cellosaurus).

In [14]:
geo_info = pd.read_csv(f'{NOMEN}/10_GEOInfo.txt', sep='\t')
geo_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 3267 entries, 0 to 3266
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Geo_accession        3267 non-null   str    
 1   CEL_file_names       3267 non-null   str    
 2   title                2329 non-null   str    
 3   status               2329 non-null   str    
 4   submission_date      2329 non-null   str    
 5   last_update_date     2329 non-null   str    
 6   type                 2329 non-null   str    
 7   channel_count        2329 non-null   float64
 8   source_name_ch1      2329 non-null   str    
 9   organism_ch1         2329 non-null   str    
 10  characteristics_ch1  2329 non-null   str    
 11  platform_id          2329 non-null   str    
 12  contact_country      2329 non-null   str    
 13  contact_institute    2329 non-null   str    
 14  GSE_ID               2329 non-null   str    
 15  GSE_filename         2329 non-null   str    
 16 

In [15]:
geo_info.head()

,Geo_accession,CEL_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_institute,GSE_ID,GSE_filename,cell_line,disease,origin,Cellosaurus_ID,Cellline,Matching_Type,cell_line_Trimmed
0,GSM101610,GSM101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_0131,A-172,Cello GEO GSM,NaN
1,GSM101615,GSM101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_0393,LN-229,Cello GEO GSM,NaN
2,GSM101616,GSM101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_0393,LN-229,Cello GEO GSM,NaN
3,GSM101667,GSM101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_1715,SW1088,Cello GEO GSM,NaN
4,GSM101668,GSM101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,CVCL_1715,SW1088,Cello GEO GSM,NaN


In [16]:
print(f"GEO samples: {len(geo_info):,}")
print(f"Samples successfully matched to a known cell line: {geo_info['Cellosaurus_ID'].notna().sum():,}")
geo_info['Matching_Type'].value_counts()

GEO samples: 3,267
Samples successfully matched to a known cell line: 3,159


Matching_Type
Cello GEO GSM              1191
Cello Cell line Name        905
Cello Cell line Synonym     528
Cello match alphanum         57
Name: count, dtype: int64

**Reading the output:** A meaningful fraction of GEO samples are unmatched (`Cellosaurus_ID` is null) — meaning the cell line name in the original experiment couldn't be confidently resolved. This is a **data quality consideration**: GEO-derived expression data is valuable for coverage but noisier in terms of cell line identity than DepMap/HPA.

---
# 2. Gene Expression (Transcriptomics) — "Which genes are switched on?"

## Why gene expression matters

Every cell carries the *same* DNA (the "recipe book"), but different cells use different *recipes* at different times — this is gene **expression**. Measuring which genes are highly expressed in a cell line tells us which biological pathways are active. For drug discovery, this is crucial: if a drug targets protein X, we want a cell line where the gene encoding X is actively expressed (otherwise there's nothing for the drug to act on, and any effect we measure would be misleading).

**Common units you'll see:**
- **TPM** (Transcripts Per Million) — a normalised measure of how many RNA copies of a gene exist per million total RNA molecules, making expression comparable across samples.
- **Log₂(TPM+1)** — TPM values are often log-transformed because gene expression spans many orders of magnitude; log-scaling makes the data easier for statistics/ML to handle.

These files are **large** (\~1 GB each for files 1–3), so below we load only small previews/samples rather than the full tables, in line with the "don't read every cell" guidance.

## 2.1 — File 1: HPA RNA Cell Line Expression

**What it is:** RNA expression (in TPM) for many genes, across ~1,200 cell lines, from the Human Protein Atlas. The table is in **long format**: one row per (gene, cell line) combination.

**Why it matters:** Broad coverage of cell lines (1,206!) makes this a great source for *screening* — quickly checking whether a gene of interest is expressed in a candidate cell line, across a huge panel.

In [17]:
# Large file (~1 GB) -> read only a sample of rows to inspect structure
hpa_rna_sample = pd.read_csv(f'{GENE_EXPR}/1_4_hpa_rna_celline.tsv', sep='\t', nrows=200_000)
hpa_rna_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Gene       200000 non-null  str    
 1   Gene name  200000 non-null  str    
 2   Cell line  200000 non-null  str    
 3   TPM        200000 non-null  float64
 4   pTPM       200000 non-null  float64
 5   nTPM       200000 non-null  float64
dtypes: float64(3), str(3)
memory usage: 9.2 MB


In [18]:
hpa_rna_sample.head()

,Gene,Gene name,Cell line,TPM,pTPM,nTPM
0,ENSG00000000003,TSPAN6,143B,22.0,27.6,25.9
1,ENSG00000000003,TSPAN6,22Rv1,2.8,3.6,2.7
2,ENSG00000000003,TSPAN6,23132/87,6.2,7.5,7.5
3,ENSG00000000003,TSPAN6,253J,14.2,18.7,25.4
4,ENSG00000000003,TSPAN6,253J-BV,13.0,17.1,18.5


In [19]:
# Within this 200k-row sample alone, how many distinct genes and cell lines appear?
print(f"Unique genes (in sample): {hpa_rna_sample['Gene'].nunique()}")
print(f"Unique cell lines (in sample): {hpa_rna_sample['Cell line'].nunique()}")
hpa_rna_sample[['TPM', 'pTPM', 'nTPM']].describe()

Unique genes (in sample): 166
Unique cell lines (in sample): 1206


,TPM,pTPM,nTPM
count,200000.000000,200000.000000,200000.000000
mean,27.539863,35.028917,34.926794
std,74.210506,94.415413,101.510353
min,0.000000,0.000000,0.000000
25%,0.400000,0.500000,0.500000
50%,8.100000,10.300000,10.400000
75%,26.200000,33.400000,32.800000
max,11403.500000,14772.700000,17961.700000


**Reading the output:** Even a 200k-row slice already spans all 1,206 cell lines — confirming the long/tidy format (one gene's expression repeated across every cell line before moving to the next gene). `nTPM` (normalised TPM) is the column most directly comparable across cell lines and is likely the one we'll use downstream.

*Note on `pTPM`*: protein-coding TPM — TPM re-normalised over protein-coding genes only.

## 2.2 — File 2: DepMap Omics Expression (TPM, log₂)

**What it is:** RNA-seq expression for **~1,500 cell lines x ~19,000 genes**, in **wide format** — one row per cell line, one column per gene, values are `log2(TPM+1)`. Cell lines are identified by their DepMap `ProfileID` (`PR-xxxxxx`).

**Why it matters:** This is DepMap's flagship expression dataset, generated with a single, consistent pipeline across all cell lines — making it the most *internally consistent* expression resource we have (compare to GEO, which aggregates many independent experiments). This consistency makes it a strong candidate as the primary expression feature matrix for modelling.

In [20]:
# Extremely wide file (~54k columns!) -> read only a handful of columns to inspect
depmap_cols = pd.read_csv(f'{GENE_EXPR}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv', nrows=0).columns
print(f"Total columns: {len(depmap_cols):,} (1 ID column + ~{len(depmap_cols)-1:,} genes)")
print("First 5 gene columns:", list(depmap_cols[1:6]))

Total columns: 53,962 (1 ID column + ~53,961 genes)
First 5 gene columns: ['TSPAN6 (ENSG00000000003)', 'TNMD (ENSG00000000005)', 'DPM1 (ENSG00000000419)', 'SCYL3 (ENSG00000000457)', 'C1orf112 (ENSG00000000460)']


In [21]:
depmap_expr_sample = pd.read_csv(
    f'{GENE_EXPR}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv',
    usecols=list(depmap_cols[:6])
)
depmap_expr_sample.head()

,Unnamed: 0,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460)
0,PR-AdBjpG,4.331992,0.000000,7.364660,2.792855,4.471187
1,PR-I2AzwG,4.567424,0.584963,7.106641,2.543496,3.504620
2,PR-5ekAAC,3.150560,0.000000,7.379118,2.333424,4.228049
3,PR-I21681,5.085340,0.000000,7.154211,2.545968,3.084064
4,PR-i9DRP1,6.729417,0.000000,6.537917,2.456806,3.867896


In [22]:
print(f"Number of cell line profiles (rows): {len(depmap_expr_sample):,}")
depmap_expr_sample.iloc[:, 1:].describe()

Number of cell line profiles (rows): 1,495


,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460)
count,1495.000000,1495.000000,1495.000000,1495.000000,1495.000000
mean,3.395123,0.076698,6.520645,2.366344,3.681424
std,1.631365,0.372394,0.644190,0.540800,0.793538
min,0.000000,0.000000,3.655352,0.594549,0.056584
25%,2.908813,0.000000,6.114992,2.007196,3.238022
50%,3.820690,0.000000,6.504779,2.336283,3.751678
75%,4.447249,0.000000,6.934045,2.670160,4.192589
max,8.132680,5.251340,9.175250,4.747387,5.972463


**Reading the output:** Each row is a `ProfileID` (`PR-xxxxxx`) — to know *which cell line* this is, we'll join via **File 8 (DepMap Omics Profiles)** to get the `ModelID`, then **File 9 (Sample Info)** to get the human-readable name and cancer type. Gene column names encode both the gene symbol and its Ensembl ID, e.g. `SCYL3 (ENSG00000000457)` — handy because it avoids ambiguity between genes with similar names.

## 2.3 — File 3: GEO Expression

**What it is:** Gene expression values (likely microarray/RNA-seq intensity, not TPM) for ~3,267 GEO samples (`GSMxxxxx`), in wide format — one row per gene (Ensembl ID), one column per sample.

**Why it matters:** Adds breadth — many more *samples* (experimental conditions/replicates), potentially covering cell lines or treatment conditions absent from DepMap. But, as seen in File 10, a chunk of these samples couldn't be confidently matched to a known cell line, so this dataset needs careful filtering before use.

In [23]:
# ~1 GB, 3268 columns -> peek at column count and a few rows/cols only
geo_expr_cols = pd.read_csv(f'{GENE_EXPR}/3_GEOexpression.txt', sep='\t', nrows=0).columns
print(f"Total columns: {len(geo_expr_cols):,} (1 Gene column + ~{len(geo_expr_cols)-1:,} GEO samples)")

geo_expr_sample = pd.read_csv(f'{GENE_EXPR}/3_GEOexpression.txt', sep='\t', usecols=list(geo_expr_cols[:6]))
geo_expr_sample.head()

Total columns: 3,268 (1 Gene column + ~3,267 GEO samples)


,Gene,GSM101610,GSM101615,GSM101616,GSM101667,GSM101668
0,ENSG00000000003,33.615700,553.249756,540.452209,599.431152,625.242737
1,ENSG00000000005,40.925682,31.327406,33.934967,34.213123,32.466286
2,ENSG00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727
3,ENSG00000000457,58.934814,95.068222,94.900459,51.810070,52.327530
4,ENSG00000000460,136.418900,257.929169,271.317230,162.090073,160.913849


In [24]:
print(f"Number of genes (rows): {len(geo_expr_sample):,}")
geo_expr_sample.iloc[:, 1:].describe()

Number of genes (rows): 19,914


,GSM101610,GSM101615,GSM101616,GSM101667,GSM101668
count,19914.000000,19914.000000,19914.000000,19914.000000,19914.000000
mean,307.754267,309.811280,319.028222,301.564948,315.117556
std,792.613875,823.046166,843.193865,787.452648,821.449956
min,11.182878,12.025491,11.774850,10.843905,11.584125
25%,54.606677,54.020123,53.232108,56.653185,54.587312
50%,104.870041,103.914730,102.965572,105.912521,104.478802
75%,262.564415,260.926208,266.130234,254.953556,265.424553
max,20061.158203,22867.765625,21479.978516,21143.740234,20322.869141


## 2.4 — File 4: Harmonized MS CCLE (Gygi Lab Proteomics)

**What it is:** **Proteomics** data — not RNA, but actual **protein abundance**, measured by mass spectrometry (MS), for ~375 CCLE cell lines (`ACH-xxxxxx`) across ~12,558 proteins.

**Why it matters — and why it's different from the rest:** RNA expression tells us a gene is "switched on", but the actual drug target is usually a **protein**. RNA and protein levels don't always correlate perfectly (a gene can be transcribed into RNA but not efficiently translated into protein, or the protein could be rapidly degraded). This dataset lets us check that a gene of interest is expressed *at the protein level* too — a stronger signal for drug-target relevance. Values here are typically standardised/relative abundances (can be negative), unlike TPM which is non-negative.

In [25]:
ms_ccle = pd.read_csv(f'{GENE_EXPR}/4_Harmonized_MS_CCLE_Gygi_subsetted.csv')
ms_ccle.info(max_cols=5)

<class 'pandas.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Columns: 12559 entries, Unnamed: 0 to Q9Y2L9-2 (LRCH1)
dtypes: float64(12558), str(1)
memory usage: 35.9 MB


In [26]:
ms_ccle.iloc[:, :5].head()

,Unnamed: 0,A0AV96 (RBM47),A0AVF1 (IFT56),A0AVG3 (TSNARE1),A0AVI4 (TMEM129)
0,ACH-000849,0.358567,-0.172066,NaN,NaN
1,ACH-000441,-1.112410,0.339446,NaN,NaN
2,ACH-000248,0.855575,-0.181171,NaN,NaN
3,ACH-000684,0.061377,-0.341233,NaN,NaN
4,ACH-000856,0.284258,-0.059558,NaN,NaN


In [27]:
print(f"Cell lines (rows): {ms_ccle.shape[0]}")
print(f"Proteins (columns): {ms_ccle.shape[1] - 1}")
print(f"Overall missing-value rate: {ms_ccle.iloc[:, 1:].isna().mean().mean():.1%}")

Cell lines (rows): 375
Proteins (columns): 12558


Overall missing-value rate: 27.8%


**Reading the output:** Note the much smaller cell-line count (~375) compared to expression data (~1,500-1,800) — proteomics is expensive and slower to run, so far fewer cell lines have been profiled this way. Also note the missing-value rate: mass spec doesn't reliably detect every protein in every sample, so `NaN` here often means "not detected", not "zero abundance". This matters for preprocessing — we cannot blindly fill these with 0.

---
# 3. Gene Properties (Genomics) — "What's structurally different about this cell line's DNA?"

## Why genomic alterations matter

While gene expression tells us *how much* of a gene's product is being made, genomics tells us whether the gene itself is **damaged, altered, or rearranged**. Cancer is fundamentally a disease of the genome: mutations can turn a normal gene into an oncogene (cancer-driving) or disable a tumour-suppressor gene. Many modern cancer drugs are designed to specifically target cells carrying a *particular* mutation (this is the basis of "precision medicine", e.g. EGFR-mutant lung cancer drugs).

So: if a researcher wants to test a drug designed for tumours with a specific mutation, we need to be able to search **which cell lines carry that mutation** — that's what these two files provide.

## 3.1 — File 5: Omics Fusion (Gene Fusions)

**What it is:** A catalogue of **gene fusions** detected by RNA sequencing — events where two separate genes become abnormally joined into one (often due to a chromosomal rearrangement). Each row is one fusion event detected in one sequencing sample (`SequencingID` / `ModelID`).

**Why it matters:** Gene fusions are some of the most famous cancer drivers (e.g. *BCR-ABL* in chronic myeloid leukaemia, the target of the drug imatinib/Gleevec). If our research question involves a fusion-driven cancer, this is the file that tells us which cell lines carry that exact fusion, with supporting evidence (read counts, confidence level).

In [28]:
fusions = pd.read_csv(f'{GENE_PROP}/5_OmicsFusionFilteredSupplementary.csv')
fusions.info(max_cols=10)

<class 'pandas.DataFrame'>
RangeIndex: 184237 entries, 0 to 184236
Columns: 30 entries, Unnamed: 0 to direction2
dtypes: float64(1), int64(8), str(21)
memory usage: 42.2 MB


In [29]:
fusions[['ModelID', 'CanonicalFusionName', 'confidence', 'TotalReadsSupportingFusion', 'FFPM']].head()

,ModelID,CanonicalFusionName,confidence,TotalReadsSupportingFusion,FFPM
0,ACH-001113,DLG1--SERPINI1,high,203,4.279581
1,ACH-001113,ADAM17--ITGB1BP1,high,138,2.909272
2,ACH-001113,ADAM17--ITGB1BP1,medium,57,1.201656
3,ACH-001113,ADAM17--ITGB1BP1,low,50,1.054084
4,ACH-001113,ADAM17--ITGB1BP1,low,22,0.463797


In [30]:
print(f"Total fusion events: {len(fusions):,}")
print(f"Cell lines with at least one fusion: {fusions['ModelID'].nunique():,}")
fusions['confidence'].value_counts()

Total fusion events: 184,237
Cell lines with at least one fusion: 1,699


confidence
low       76217
high      55521
medium    52499
Name: count, dtype: int64

**Reading the output:** Note the `confidence` column — fusion detection from sequencing reads is algorithmic and can produce false positives, so results are graded (e.g. high/low confidence). For our pipeline, we'd likely filter to high-confidence fusions only when using this as a feature/justification source.

## 3.2 — File 6: Omics Somatic Mutations Profile

**What it is:** The big one — a catalogue of **somatic mutations** (DNA changes acquired during the cell's life, not inherited) detected across cell lines via whole exome/genome sequencing. Each row is one mutation in one sample (`ProfileID`), with ~70 columns of annotation: the gene affected (`HugoSymbol`), the type of change (`VariantType`, `ProteinChange`), predicted functional impact (`VepImpact`, `Sift`, `Polyphen`, `AMPathogenicity`), and whether it's a known cancer driver (`OncogeneHighImpact`, `TumorSuppressorHighImpact`, `Hotspot`, `CivicID`).

**Why it matters:** This is arguably the **most direct "why" signal** for cell line selection. If a researcher's drug targets, e.g., *KRAS G12C*-mutant tumours, this file lets us filter directly for cell lines whose `HugoSymbol == 'KRAS'` and `ProteinChange` matches that specific mutation — an extremely concrete, biologically-grounded justification.

In [31]:
# Very large file (~500MB, 70 cols) -> inspect structure with a sample first
mutations_sample = pd.read_csv(f'{GENE_PROP}/6_OmicsSomaticMutationsProfile.csv', nrows=100_000)
mutations_sample.info(max_cols=10)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 70 entries, Chrom to EntrezGeneID
dtypes: bool(6), float64(17), int64(4), str(43)
memory usage: 49.4 MB


C:\Users\ksstu\AppData\Local\Temp\ipykernel_24396\464706879.py:2: DtypeWarning: Columns (0: CivicDescription, 1: PharmgkbId, 2: DidaID, 3: DidaName, 4: GwasDisease, 5: GtexGene) have mixed types. Specify dtype option on import or set low_memory=False.
  mutations_sample = pd.read_csv(f'{GENE_PROP}/6_OmicsSomaticMutationsProfile.csv', nrows=100_000)


In [32]:
key_cols = ['ProfileID', 'HugoSymbol', 'ProteinChange', 'VariantType', 'VepImpact',
             'OncogeneHighImpact', 'TumorSuppressorHighImpact', 'Hotspot']
mutations_sample[key_cols].head()

,ProfileID,HugoSymbol,ProteinChange,VariantType,VepImpact,OncogeneHighImpact,TumorSuppressorHighImpact,Hotspot
0,PR-t8SaQo,FAM87B,NaN,SNV,HIGH,False,False,False
1,PR-lhMBt6,LINC01128,NaN,SNV,HIGH,False,False,False
2,PR-uKiczK,SAMD11,p.A27K,substitution,MODERATE,False,False,False
3,PR-sxFiuq,SAMD11,p.L76V,SNV,MODERATE,False,False,False
4,PR-DNEoiz,SAMD11,p.P107S,SNV,MODERATE,False,False,False


In [33]:
print(f"Profiles in this sample: {mutations_sample['ProfileID'].nunique()}")
print(f"Distinct genes affected (in sample): {mutations_sample['HugoSymbol'].nunique():,}")
mutations_sample['VariantType'].value_counts()

Profiles in this sample: 919
Distinct genes affected (in sample): 5,932


VariantType
SNV             85562
deletion         7030
substitution     5151
insertion        2257
Name: count, dtype: int64

In [34]:
mutations_sample['VepImpact'].value_counts(dropna=False)

VepImpact
MODERATE    83033
HIGH        16966
MODIFIER        1
Name: count, dtype: int64

**Reading the output:** Even a 100k-row sample contains a huge diversity of mutated genes — this reflects the reality that any tumour genome has thousands of mutations, most of which are biologically irrelevant "passenger" mutations. The `VepImpact` (variant effect predictor) and driver-gene flags (`OncogeneHighImpact`, `Hotspot`, etc.) are how we separate the meaningful "driver" mutations from the noise. For modelling, we'll almost certainly aggregate this table per cell line (e.g. "does cell line X have a high-impact mutation in gene Y?") rather than use raw rows.

---
# 4. Non-Gene-Expression (Other Omics Layers)

## Why look beyond genes?

Genes and proteins don't act alone — a cell's behaviour is also shaped by its **metabolism** (the small molecules it produces and consumes), its **regulatory RNAs** (microRNAs, which fine-tune gene expression), and its overall **genome stability**. These layers can reveal a cell line's biological state in ways gene expression alone can't — for example, a tumour's metabolic profile can hint at how it will respond to certain drug classes (e.g. drugs targeting metabolism in cancer).

## 4.1 — File 12: CCLE Metabolomics

**What it is:** Measured levels of ~225 small-molecule **metabolites** (sugars, amino acids, lipids, etc.) across CCLE cell lines, identified by `CCLE_ID` (a verbose name like `DMS53_LUNG`) and `DepMap_ID`.

**Why it matters:** Metabolite levels are a window into a cell's biochemistry — e.g. elevated lactate can indicate a cell relying heavily on a cancer-typical metabolic shortcut (the "Warburg effect"). This can support justifications around metabolism-targeting drugs, and also gives us a *third* identifier format (`CCLE_ID`, which embeds the tissue name) to cross-reference.

In [35]:
metabolomics = pd.read_csv(f'{NON_GENE_EXPR}/12_CCLE_metabolomics_20190502.csv')
metabolomics.info(max_cols=10)

<class 'pandas.DataFrame'>
RangeIndex: 928 entries, 0 to 927
Columns: 227 entries, CCLE_ID to C58:6 TAG
dtypes: float64(225), str(2)
memory usage: 1.6 MB


In [36]:
metabolomics.iloc[:, :6].head()

,CCLE_ID,DepMap_ID,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate
0,DMS53_LUNG,ACH-000698,6.112727,6.034198,5.896896,6.000532
1,SW1116_LARGE_INTESTINE,ACH-000489,5.577413,5.727045,5.111468,6.073250
2,NCIH1694_LUNG,ACH-000431,5.886398,5.574881,5.541259,5.848375
3,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,ACH-000707,5.770030,6.099229,6.233259,5.543495
4,HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,ACH-000509,5.480683,5.469742,6.509397,6.251005


In [37]:
print(f"Cell lines (rows): {metabolomics.shape[0]}")
print(f"Metabolites measured (columns): {metabolomics.shape[1] - 2}")
metabolomics.iloc[:, 2:].describe().iloc[:, :4]

Cell lines (rows): 928
Metabolites measured (columns): 225


,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate
count,928.000000,928.000000,928.000000,928.000000
mean,5.938204,5.870931,5.924995,5.955254
std,0.312954,0.316493,0.521475,0.335108
min,4.933644,4.891322,4.051158,4.960722
25%,5.750317,5.659425,5.545633,5.697951
50%,5.905811,5.886291,5.880360,5.974359
75%,6.083362,6.086525,6.240539,6.195044
max,7.916397,6.730925,7.442171,7.222248


## 4.2 — File 13: CCLE miRNA

**What it is:** Expression levels of **microRNAs** (short RNA molecules, ~22 nucleotides, that regulate gene expression by binding to messenger RNAs) across ~954 CCLE cell lines. The file is in **GCT format** — a standard genomics format with 2 metadata header lines, then a table where rows are miRNAs (`Name`/`Description`) and columns are cell lines (named like `DMS53_LUNG`).

**Why it matters:** microRNAs act as "dimmer switches" for gene expression — a single miRNA can simultaneously suppress many target genes. They're increasingly studied as both biomarkers (e.g. for cancer subtype) and therapeutic targets. Including miRNA data lets our pipeline capture *regulatory* context that raw gene expression misses.

In [38]:
# GCT format: first line = version, second line = dimensions, third line = header
with open(f'{NON_GENE_EXPR}/13_CCLE_miRNA_20181103.gct') as f:
    version_line = f.readline().strip()
    dims_line = f.readline().strip()
print("Version line:", version_line)
print("Dimensions (rows, cols):", dims_line)

mirna = pd.read_csv(f'{NON_GENE_EXPR}/13_CCLE_miRNA_20181103.gct', sep='\t', skiprows=2)
mirna.iloc[:, :5].head()

Version line: #1.2
Dimensions (rows, cols): 734	954


,Name,Description,DMS53_LUNG,SW1116_LARGE_INTESTINE,NCIH1694_LUNG
0,nmiR00001.1,hsa-let-7a,4362.58,5191.50,24991.05
1,nmiR00002.1,hsa-let-7b,187.44,868.22,5066.09
2,nmiR00003.1,hsa-let-7c,267.03,244.88,818.49
3,nmiR00004.1,hsa-let-7d,868.11,556.55,3661.86
4,nmiR00005.1,hsa-let-7e,1.04,92.02,942.62


In [39]:
print(f"microRNAs measured (rows): {mirna.shape[0]}")
print(f"Cell lines (columns): {mirna.shape[1] - 2}")
mirna.iloc[:, 2:].describe().iloc[:, :4]

microRNAs measured (rows): 734
Cell lines (columns): 954


,DMS53_LUNG,SW1116_LARGE_INTESTINE,NCIH1694_LUNG,P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
count,734.000000,734.000000,734.000000,734.000000
mean,212.437575,166.253174,447.604482,359.542711
std,1131.780168,680.735430,3311.920850,2005.855335
min,1.040000,1.480000,1.930000,1.900000
25%,10.480000,17.810000,15.520000,9.510000
50%,15.710000,28.200000,25.210000,20.930000
75%,32.460000,50.460000,59.637500,55.180000
max,22579.410000,9327.780000,77671.050000,26251.360000


**Reading the output:** Notice this table is **transposed** relative to most of our other files — here, cell lines are *columns* and miRNAs are *rows*. We'll need to transpose this before merging it with the rest of our (cell-line-as-row) data.

## 4.3 — File 14: Omics Global Signatures

**What it is:** A compact table of **genome-wide instability scores** per sequencing profile: `MSIScore` (microsatellite instability — a marker of DNA repair deficiency), `LoHFraction` (fraction of the genome showing loss of heterozygosity), `WGD` (whether a whole-genome duplication occurred), `CIN` (chromosomal instability score), `Ploidy` (how many copies of the genome the cell carries — normal human cells are diploid, i.e. 2), and `Aneuploidy` (count of chromosome arms with abnormal copy number).

**Why it matters:** These are "big picture" genome health scores. High microsatellite instability (MSI-high), for instance, is a clinically-used biomarker — MSI-high tumours often respond well to immunotherapy. Including these scores lets our justification step say things like "this cell line is MSI-high, making it a good model for immune-checkpoint-inhibitor studies."

In [40]:
global_sig = pd.read_csv(f'{NON_GENE_EXPR}/14_OmicsGlobalSignatures.csv')
global_sig.info()

<class 'pandas.DataFrame'>
RangeIndex: 3021 entries, 0 to 3020
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              3021 non-null   int64  
 1   SequencingID            3021 non-null   str    
 2   ModelID                 3021 non-null   str    
 3   ModelConditionID        3021 non-null   str    
 4   IsDefaultEntryForModel  3021 non-null   str    
 5   IsDefaultEntryForMC     3021 non-null   str    
 6   MSIScore                3021 non-null   float64
 7   LoHFraction             2582 non-null   float64
 8   WGD                     2582 non-null   float64
 9   CIN                     2582 non-null   float64
 10  Ploidy                  2582 non-null   float64
 11  Aneuploidy              2582 non-null   float64
dtypes: float64(6), int64(1), str(5)
memory usage: 283.3 KB


In [41]:
global_sig.head()

,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy
0,0,CDS-00Nrci,ACH-000839,MC-000839-krru,Yes,Yes,3.68,0.107443,1.0,0.502634,3.158291,20.0
1,1,CDS-051xn7,ACH-000041,MC-000041-uPBf,Yes,Yes,2.21,0.130089,1.0,0.523865,3.236089,19.0
2,2,CDS-099jzP,ACH-002046,MC-002046-oaX8,Yes,Yes,2.87,0.222342,1.0,0.679772,3.326715,30.0
3,3,CDS-0A4mDu,ACH-002048,MC-002048-52d6,Yes,Yes,2.48,NaN,NaN,NaN,NaN,NaN
4,4,CDS-0Eax8o,ACH-000042,MC-000042-eOnX,Yes,Yes,2.07,NaN,NaN,NaN,NaN,NaN


In [42]:
print(f"Profiles (rows): {len(global_sig):,}")
print(f"Distinct cell lines (ModelID): {global_sig['ModelID'].nunique():,}")
global_sig[['MSIScore', 'LoHFraction', 'CIN', 'Ploidy', 'Aneuploidy']].describe()

Profiles (rows): 3,021
Distinct cell lines (ModelID): 1,955


,MSIScore,LoHFraction,CIN,Ploidy,Aneuploidy
count,3021.000000,2582.000000,2582.000000,2582.000000,2582.000000
mean,6.390755,0.218607,0.472440,2.983539,16.735864
std,16.701642,0.163588,0.238170,0.801769,9.752613
min,0.300000,0.000000,0.000000,1.672836,0.000000
25%,1.240000,0.079719,0.272489,2.174321,8.000000
50%,2.080000,0.199456,0.540837,2.950443,19.000000
75%,3.220000,0.325502,0.663215,3.560720,24.000000
max,93.220000,0.929901,0.871745,4.994777,39.000000


---
# 5. Putting it all together — the identifier map

Across all 14 files, the same cell line can appear under several different identifiers. Here's the cheat-sheet for how everything connects:

| ID format | Example | Used in |
|---|---|---|
| `ACH-xxxxxx` (DepMap `ModelID` / `DepMap_ID`) | `ACH-000849` | Files 4, 5, 8, 9, 12, 14 |
| `PR-xxxxxx` (DepMap `ProfileID`) | `PR-AdBjpG` | Files 2, 6, 8, 14 |
| Plain cell line name | `MCF7`, `143B` | Files 1, 11 |
| `CCLE_ID` (`NAME_TISSUE`) | `DMS53_LUNG` | Files 12, 13 |
| `CVCL_xxxx` (Cellosaurus accession) | `CVCL_0030` | Files 7, 10, 11 |
| `GSMxxxxx` (GEO sample) | `GSM101610` | Files 3, 10 |

**The join path** for a typical analysis will be:

```
File 8 (ProfileID <-> ModelID)
   |        |
File 2,6,14   File 9 (ModelID -> cancer type, lineage, name)
(ProfileID)         |
                File 4,5,12 (ModelID)
                     |
                File 7 (Cellosaurus, via name/CVCL_ID -> resolves File 1, 11, 12, 13 names)
```

GEO data (Files 3, 10) sits somewhat apart — useful for extra coverage, but joined via the noisier `Matching_Type`/Cellosaurus matching in File 10.

## Summary table of all 14 files

| # | File | Rows x Cols (approx) | Primary key | Layer |
|---|---|---|---|---|
| 1 | HPA RNA cell line | ~6.6M x 6 | Gene + Cell line | Gene expression |
| 2 | DepMap Expression (TPM log) | ~1.5k x 54k | ProfileID | Gene expression |
| 3 | GEO Expression | ~63k x 3.3k | Gene (rows), GSM (cols) | Gene expression |
| 4 | Harmonized MS CCLE (Gygi proteomics) | ~375 x 12.6k | ModelID | Gene expression (protein) |
| 5 | Omics Fusion | many rows x 30 | ModelID | Gene properties |
| 6 | Omics Somatic Mutations | millions x 70 | ProfileID | Gene properties |
| 7 | Cellosaurus | ~hundreds of k x 17 | CVCL accession | Nomenclature |
| 8 | DepMap Omics Profiles | 3,830 x 5 | ProfileID, ModelID | Nomenclature (bridge) |
| 9 | DepMap Sample Info | 1,840 x 29 | DepMap_ID | Nomenclature |
| 10 | GEO Info | ~3.3k x 23 | Geo_accession | Nomenclature |
| 11 | HPA Cell Line Description | 1,206 x 7 | Cell line | Nomenclature |
| 12 | CCLE Metabolomics | ~930 x 227 | CCLE_ID, DepMap_ID | Other omics |
| 13 | CCLE miRNA | 734 x 956 | Name (rows), CCLE_ID (cols) | Other omics |
| 14 | Omics Global Signatures | 3,021 x 12 | ModelID, SequencingID | Other omics |

## Where to next

Now that everyone has the full picture, the natural next steps for the **preprocessing pipeline** (`preprocessing/`) are:

1. **Build a master cell-line index** — one row per `ModelID`, with all known aliases (`ProfileID`, `CCLE_ID`, plain name, `CVCL_ID`) resolved via Files 7–11.
2. **Aggregate the per-mutation / per-fusion tables (5, 6)** into per-cell-line features (e.g. "has high-impact mutation in gene X: yes/no").
3. **Decide on a primary expression source** — DepMap (File 2) is the most internally consistent; HPA (File 1) and GEO (File 3) extend coverage but need careful matching.
4. **Reshape File 13 (miRNA)** so cell lines are rows, consistent with everything else.
5. **Join Files 12 & 14** directly onto the master index via `ModelID`.

Each of these decisions should be documented with its biological rationale — remember, the project's success criterion isn't just *which* cell line is recommended, but *why*.